# Lab 4 Task 2: OFDM Phase Offset

Complete the OFDM accounting and pilot phase calculation, then measure BER and FER from 0 through 90 degrees.


In [ ]:
from pathlib import Path
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image, display

def locate_lab_root():
    for base in (Path.cwd(), *Path.cwd().parents):
        if (base / "support" / "lab4_tools.py").is_file():
            return base
    raise FileNotFoundError("Open this notebook from its Lab 4 folder.")

LAB_ROOT = locate_lab_root()
sys.path.insert(0, str(LAB_ROOT / "support"))
WORK_DIR = LAB_ROOT / "work"
FIGURE_DIR = WORK_DIR / "figures"
WAVEFORM_DIR = WORK_DIR / "waveforms"
CAPTURE_DIR = Path(os.environ.get("ECE7377_LAB4_CAPTURE_DIR", WORK_DIR / "captures"))
for path in (FIGURE_DIR, WAVEFORM_DIR, CAPTURE_DIR):
    path.mkdir(parents=True, exist_ok=True)
print("Lab root:", LAB_ROOT)
print("Capture directory:", CAPTURE_DIR)


## OFDM calculations


In [ ]:
nfft = 64
cyclic_prefix_samples = 16
pilot_subcarriers = 4
qpsk_bits_per_symbol = 2

# To do: derive the active-carrier and frame counts.
active_subcarriers = ...
data_subcarriers = ...
bits_per_frame = ...
time_samples_per_frame = ...

print("data subcarriers:", data_subcarriers)
print("bits per frame:", bits_per_frame)
print("time samples per frame:", time_samples_per_frame)


In [ ]:
assert data_subcarriers == 48
assert bits_per_frame == 96
assert time_samples_per_frame == 80
print("OFDM accounting passed.")


## Pilot phase correction check


In [ ]:
known_pilots = np.ones(4, dtype=complex)
received_pilots = known_pilots*np.exp(1j*np.deg2rad(30))

# To do: estimate and remove the common pilot phase.
phase_hat_rad = ...
corrected_pilots = ...

print("estimated phase (degrees):", np.rad2deg(phase_hat_rad))
print("largest corrected pilot error:", np.max(np.abs(corrected_pilots - known_pilots)))


In [ ]:
assert np.isclose(np.rad2deg(phase_hat_rad), 30.0)
assert np.max(np.abs(corrected_pilots - known_pilots)) < 1e-12
print("Pilot correction passed.")


## Full phase sweep


In [ ]:
from lab4_tools import ofdm_reference, plot_ofdm_phase_study

result = ofdm_reference(
    phase_degrees=(0, 30, 60, 90),
    frames=1000,
    ebn0_db=10.0,
    message="hello world",
)
figure_paths = plot_ofdm_phase_study(result, FIGURE_DIR)
for path in figure_paths:
    display(Image(filename=path))

print("phase   BER       FER     phase estimate   corrected BER   corrected FER")
for row in result["rows"]:
    print(
        f"{row['phase_degrees']:>3d}  {row['ber']:.6f}  {row['fer']:.3f}  "
        f"{row['estimated_phase_mean_degrees']:>9.3f} deg  "
        f"{row['corrected_ber']:.7f}       {row['corrected_fer']:.3f}"
    )


## Report work

1. Report BER and FER before correction at all four phase offsets.
2. Explain why FER may approach one while BER remains much smaller.
3. Report the mean pilot phase estimate and corrected error rates.
4. Explain the difference between constant phase offset and CFO.
5. Include both saved figures and the result table.
